#  Racial Bias Detection — Modeling

####  Load Dataset

In [1]:
from datasets import load_dataset

ds = load_dataset("mediabiasgroup/mbib-base")
racial_ds = ds["racial_bias"]

df = racial_ds.to_pandas()

#### Data Cleaning & Text Normalization

In [2]:
import pandas as pd
import numpy as np

# Create a copy for cleaning
clean_df = df.copy()

# Normalize text for duplicate detection
clean_df["normalized_text"] = (
    clean_df["text"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Identify conflicting labels
label_counts = (
    clean_df.groupby("normalized_text")["label"]
    .nunique()
)

conflicting_keys = label_counts[
    label_counts > 1
].index

# Remove conflicting text groups
clean_df = clean_df[
    ~clean_df["normalized_text"].isin(conflicting_keys)
].copy()

# Remove consistent normalized duplicates
clean_df = clean_df.drop_duplicates(
    subset=["normalized_text"],
    keep="first"
).copy()

# Validate dataset
assert clean_df["label"].isin([0, 1]).all()
assert clean_df["normalized_text"].is_unique
assert clean_df["text"].notna().all()

print("Original dataset:", df.shape)
print("Cleaned dataset:", clean_df.shape)

display(clean_df["label"].value_counts())

Original dataset: (9788, 4)
Cleaned dataset: (9784, 5)


label
0    4893
1    4891
Name: count, dtype: int64

In [3]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd
import numpy as np

# 1. Keep records sharing the same source-specific ID in a single split.
#    This prevents the previously observed Source 75 ID overlap.
#    Unique rows with genuinely missing IDs receive their own group.
required = {"dataset_id", "id", "text", "label", "normalized_text"}
missing = required - set(clean_df.columns)
if missing:
    raise ValueError(f"Required columns missing for leak-safe splitting: {sorted(missing)}")

source = clean_df["dataset_id"].astype("string").fillna("<missing-source>")
record_id = clean_df["id"].astype("string")
valid_id = record_id.notna() & record_id.fillna("").str.strip().ne("")
group_keys = "source=" + source + "|id=" + record_id.fillna("")
group_keys.loc[~valid_id] = [
    f"missing-id-row={idx}" for idx in clean_df.index[~valid_id]
]
clean_df["split_group"] = pd.factorize(group_keys)[0]

# 2. Outer group-aware split: approximately 80% train / 20% holdout.
#    Unlike StratifiedShuffleSplit, related IDs cannot be split apart.
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, temp_idx = next(outer_cv.split(
    clean_df["text"], clean_df["label"], groups=clean_df["split_group"]
))
train_df = clean_df.iloc[train_idx].copy()
temp_df = clean_df.iloc[temp_idx].copy()

# 3. Inner group-aware split: divide holdout into Validation / Test.
inner_cv = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)
val_idx, test_idx = next(inner_cv.split(
    temp_df["text"], temp_df["label"], groups=temp_df["split_group"]
))
val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# 4. Verify split sizes and label distributions.
#    Group-aware splits may differ slightly from an exact 80/10/10 ratio.
for name, subset in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    assert subset["label"].nunique() == 2, f"{name} contains only one class."
    print(
        f"{name} set: {len(subset)} rows "
        f"({len(subset)/len(clean_df)*100:.1f}%) | "
        f"Labels distribution: {subset['label'].value_counts(normalize=True).round(3).to_dict()}"
    )
print("Unique source-specific groups:", clean_df["split_group"].nunique())

Train set: 7837 rows (80.1%) | Labels distribution: {1: 0.501, 0: 0.499}
Validation set: 1008 rows (10.3%) | Labels distribution: {0: 0.514, 1: 0.486}
Test set: 939 rows (9.6%) | Labels distribution: {1: 0.509, 0: 0.491}
Unique source-specific groups: 5029


In [4]:
# Data leakage checks: text, source-specific IDs, and group IDs.

def check_overlap(col):
    train = set(train_df[col])
    val = set(val_df[col])
    test = set(test_df[col])

    overlaps = {
        "Train / Validation": len(train & val),
        "Train / Test": len(train & test),
        "Validation / Test": len(val & test),
    }
    print(f"\n{col} overlap:")
    for pair, count in overlaps.items():
        print(f"{pair}: {count}")
        assert count == 0, f"Leakage: {count} shared {col} in {pair}"

check_overlap("normalized_text")
check_overlap("split_group")

# Explicit Source 75 check, matching the previous investigation.
if {"dataset_id", "id"}.issubset(clean_df.columns):
    source75_sets = [
        set(part.loc[(pd.to_numeric(part["dataset_id"], errors="coerce") == 75) & part["id"].notna(), "id"].astype(str))
        for part in (train_df, val_df, test_df)
    ]
    print("\nSource 75 shared IDs:")
    for label, a, b in [
        ("Train / Validation", 0, 1),
        ("Train / Test", 0, 2),
        ("Validation / Test", 1, 2)
    ]:
        shared = source75_sets[a] & source75_sets[b]
        print(label + ":", len(shared))
        assert not shared, f"Source 75 ID leakage in {label}"

assert len(train_df) + len(val_df) + len(test_df) == len(clean_df)
assert set(train_df.index).isdisjoint(val_df.index)
assert set(train_df.index).isdisjoint(test_df.index)
assert set(val_df.index).isdisjoint(test_df.index)
print("\nPASS: every cleaned record is assigned once, with no group/text overlap.")


normalized_text overlap:
Train / Validation: 0
Train / Test: 0
Validation / Test: 0

split_group overlap:
Train / Validation: 0
Train / Test: 0
Validation / Test: 0

Source 75 shared IDs:
Train / Validation: 0
Train / Test: 0
Validation / Test: 0

PASS: every cleaned record is assigned once, with no group/text overlap.


##  Baseline Models

###  Dummy Classifier and TF-IDF + Logistic Regression

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedGroupKFold
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

# 1. Prepare training and validation data
X_train, y_train = train_df["text"], train_df["label"]
X_val, y_val = val_df["text"], val_df["label"]

# 2. Define 3-Fold Cross-Validation strategy
cv_strategy = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

# --- A. Dummy Classifier Baseline ---
dummy_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', DummyClassifier(strategy="stratified", random_state=42))
])

dummy_pipeline.fit(X_train, y_train)
dummy_val_pred = dummy_pipeline.predict(X_val)
dummy_val_proba = dummy_pipeline.predict_proba(X_val)[:, 1]

dummy_results = {
    "Model": "Dummy Classifier",
    "ROC-AUC": roc_auc_score(y_val, dummy_val_proba),
    "PR-AUC": average_precision_score(y_val, dummy_val_proba),
    "Accuracy": accuracy_score(y_val, dummy_val_pred),
    "Precision": precision_score(y_val, dummy_val_pred, zero_division=0),
    "Recall": recall_score(y_val, dummy_val_pred, zero_division=0),
    "F1": f1_score(y_val, dummy_val_pred, zero_division=0)
}


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

SEED = 42

# 1. Define and train the Logistic Regression model with the optimal settings (C=0.5, min_df=2)
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, min_df=2, stop_words='english')),
    ('clf', LogisticRegression(C=0.5, random_state=SEED, max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)

# 2. Predict and extract probabilities
lr_val_pred = lr_pipeline.predict(X_val)
lr_val_proba = lr_pipeline.predict_proba(X_val)[:, 1]

# 3. Store the results in a dictionary for comparison
lr_results = {
    "Model": "TF-IDF + Logistic Regression",
    "ROC-AUC": roc_auc_score(y_val, lr_val_proba),
    "PR-AUC": average_precision_score(y_val, lr_val_proba),
    "Accuracy": accuracy_score(y_val, lr_val_pred),
    "Precision": precision_score(y_val, lr_val_pred, zero_division=0),
    "Recall": recall_score(y_val, lr_val_pred, zero_division=0),
    "F1": f1_score(y_val, lr_val_pred, zero_division=0)
}

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score

SEED = 42

models_to_test = {
    "current (C=1, 10k feats)": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, stop_words='english')),
        ("clf", LogisticRegression(random_state=SEED, max_iter=1000))
    ]),
    "C=0.5 (min_df=2)": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, min_df=2, stop_words='english')),
        ("clf", LogisticRegression(C=0.5, random_state=SEED, max_iter=1000))
    ]),
    "C=0.1 (min_df=3)": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, min_df=3, stop_words='english')),
        ("clf", LogisticRegression(C=0.1, random_state=SEED, max_iter=1000))
    ]),
    "C=0.05 (min_df=3)": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, min_df=3, stop_words='english')),
        ("clf", LogisticRegression(C=0.05, random_state=SEED, max_iter=1000))
    ])
}

for name, pipe in models_to_test.items():
    pipe.fit(X_train, y_train)
    tr  = roc_auc_score(y_train, pipe.predict_proba(X_train)[:, 1])
    val = roc_auc_score(y_val,   pipe.predict_proba(X_val)[:, 1])
    print(f"{name:<26} train={tr:.4f}  val={val:.4f}  gap={tr-val:+.4f}")

current (C=1, 10k feats)   train=0.9093  val=0.7990  gap=+0.1102
C=0.5 (min_df=2)           train=0.8688  val=0.7983  gap=+0.0705
C=0.1 (min_df=3)           train=0.8132  val=0.7794  gap=+0.0338
C=0.05 (min_df=3)          train=0.7930  val=0.7691  gap=+0.0239


## Approved Racial Logistic Regression — Final Evaluation & Export

The **existing Logistic Regression configuration** is retained: TF-IDF (`max_features=10000`, `min_df=2`, `stop_words='english'`) + `LogisticRegression(C=0.5, random_state=42, max_iter=1000)`.

**Required order:** Run the corrected split and leakage checks; train/validate the original LR in the cell immediately above; review the ROC-AUC and Train/Validation gap. **Do not use historical SVC/LSTM metrics as evidence for the repaired split.** Then run the following cells to train LR on **Train + Validation**, evaluate on the untouched **Test** once, and export the fitted LR + TF-IDF Pipeline for the Final Predictions notebook.

In [8]:
# Review the unchanged LR on the NEW group-safe split before final selection.
from sklearn.metrics import roc_auc_score

lr_train_auc = roc_auc_score(y_train, lr_pipeline.predict_proba(X_train)[:, 1])
lr_val_auc = roc_auc_score(y_val, lr_pipeline.predict_proba(X_val)[:, 1])

print("Approved model: TF-IDF + Logistic Regression (C=0.5)")
print(f"Train ROC-AUC:      {lr_train_auc:.4f}")
print(f"Validation ROC-AUC: {lr_val_auc:.4f}")
print(f"Train - Val gap:    {lr_train_auc - lr_val_auc:+.4f}")
print("Validation metrics:")
display(pd.DataFrame([lr_results]).round(4))

Approved model: TF-IDF + Logistic Regression (C=0.5)
Train ROC-AUC:      0.8688
Validation ROC-AUC: 0.7983
Train - Val gap:    +0.0705
Validation metrics:


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,TF-IDF + Logistic Regression,0.7983,0.7846,0.7262,0.709,0.7408,0.7246


In [9]:
# FINALIZATION: run only after accepting the LR settings based on Train/Validation.
# The held-out Test remains unused during fitting and model selection.
from sklearn.base import clone
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "models").is_dir() and (p / "notebooks").is_dir()),
    None
)
if project_root is None:
    raise FileNotFoundError("Run the notebook from inside the BiasLens project.")

model_path = project_root / "models" / "racial_bias_lr.joblib"
if model_path.exists():
    raise FileExistsError(
        f"Will not overwrite an existing Racial LR artifact: {model_path}. "
        "Review it first or deliberately choose another filename."
    )

development_df = pd.concat([train_df, val_df], ignore_index=True)
final_racial_lr = clone(lr_pipeline)  # Exactly the same approved hyperparameters
final_racial_lr.fit(development_df["text"], development_df["label"])

print("Development samples:", len(development_df))
print("Model fitted on Train + Validation only; Test still unseen.")

Development samples: 8845
Model fitted on Train + Validation only; Test still unseen.


In [10]:
# FINAL test evaluation: only run after the model and settings are fixed.
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, accuracy_score, confusion_matrix,
    classification_report
)

X_test, y_test = test_df["text"], test_df["label"]
test_proba = final_racial_lr.predict_proba(X_test)[:, 1]
test_pred = final_racial_lr.predict(X_test)

print("=== Final Racial LR Test Evaluation ===")
print(f"Test samples: {len(X_test)}")
print(f"ROC-AUC:      {roc_auc_score(y_test, test_proba):.4f}")
print(f"PR-AUC:       {average_precision_score(y_test, test_proba):.4f}")
print(f"Accuracy:     {accuracy_score(y_test, test_pred):.4f}")
print(f"Precision:    {precision_score(y_test, test_pred, zero_division=0):.4f}")
print(f"Recall:       {recall_score(y_test, test_pred, zero_division=0):.4f}")
print(f"F1:           {f1_score(y_test, test_pred, zero_division=0):.4f}")
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, zero_division=0))

# Save the same complete fitted Pipeline (vectorizer + classifier).
# Never replace any existing model without explicit review.
if model_path.exists():
    raise FileExistsError(f"Refusing to overwrite: {model_path}")
joblib.dump(final_racial_lr, model_path)
reloaded_model = joblib.load(model_path)  # Only load your own trusted artifact
np.testing.assert_allclose(
    final_racial_lr.predict_proba(X_test.iloc[:5]),
    reloaded_model.predict_proba(X_test.iloc[:5])
)
print("Saved and verified:", model_path)

=== Final Racial LR Test Evaluation ===
Test samples: 939
ROC-AUC:      0.7724
PR-AUC:       0.7830
Accuracy:     0.7082
Precision:    0.7000
Recall:       0.7469
F1:           0.7227
Confusion matrix [[TN, FP], [FN, TP]]:
[[308 153]
 [121 357]]
              precision    recall  f1-score   support

           0       0.72      0.67      0.69       461
           1       0.70      0.75      0.72       478

    accuracy                           0.71       939
   macro avg       0.71      0.71      0.71       939
weighted avg       0.71      0.71      0.71       939

Saved and verified: /Users/sarahaltaleb/Documents/GitHub/BiasLens/models/racial_bias_lr.joblib


---
## Historical experiments (preserved from the original notebook)

The following LSTM, LinearSVC, extra diagnostics and old results are kept as written by the original author. They are **not the selected Racial production model**.

**Do not use their saved output tables to describe performance after the new split.** Some of these historical cells train on Train + Validation and then report scores on Validation, which is not a valid unseen evaluation. They are retained for reference, **not part of the required LR run path**, and should **not be executed with `Run All`**.

##  LSTM Model

In [11]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import pandas as pd

# 1. Setup Vocabulary and Sequence Parameters (Fit ONLY on Training data)
VOCAB_SIZE = 10000
MAX_LENGTH = 150

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

# 2. Text to Sequences & Padding
X_train_seq = tokenizer.texts_to_sequences(train_df["text"])
X_val_seq = tokenizer.texts_to_sequences(val_df["text"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')

# 3. Build Regularized Small LSTM Architecture (To Prevent Overfitting)
model_lstm = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=64,
        mask_zero=True,
        embeddings_regularizer=l2(1e-4)
    ),
    LSTM(
        32,                          # Reduced units from 64 to 32 to lower complexity
        dropout=0.4,                 # Dropout on inputs
        recurrent_dropout=0.2        # Dropout on recurrent state
    ),
    Dropout(0.5),                    # Additional dropout layer
    Dense(1, activation='sigmoid', kernel_regularizer=l2(1e-4))
])

model_lstm.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')]
)

# 4. Early Stopping Configuration
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# 5. Train Model with Regularization
history = model_lstm.fit(
    X_train_pad, train_df["label"],
    validation_data=(X_val_pad, val_df["label"]),
    batch_size=64,
    epochs=15,
    callbacks=[early_stopping],
    verbose=1
)

# 6. Evaluate LSTM on Validation Set & Store in Structured Dictionary
val_probs_lstm = model_lstm.predict(X_val_pad).flatten()
val_preds_lstm = (val_probs_lstm >= 0.5).astype(int)

lstm_results = {
    "Model": "Small LSTM",
    "ROC-AUC": roc_auc_score(val_df["label"], val_probs_lstm),
    "PR-AUC": average_precision_score(val_df["label"], val_probs_lstm),
    "Accuracy": accuracy_score(val_df["label"], val_preds_lstm),
    "Precision": precision_score(val_df["label"], val_preds_lstm, zero_division=0),
    "Recall": recall_score(val_df["label"], val_preds_lstm, zero_division=0),
    "F1": f1_score(val_df["label"], val_preds_lstm, zero_division=0)
}

# 7. Display Results in a Clean DataFrame Table
lstm_results_df = pd.DataFrame([lstm_results])
print("\n=== Small LSTM Validation Performance Table ===")
display(lstm_results_df.round(4))

print("\nClassification Report:")
print(classification_report(val_df["label"], val_preds_lstm))

Epoch 1/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - accuracy: 0.6157 - loss: 0.6601 - roc_auc: 0.6829 - val_accuracy: 0.6667 - val_loss: 0.6654 - val_roc_auc: 0.7166
Epoch 2/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.7735 - loss: 0.5128 - roc_auc: 0.8418 - val_accuracy: 0.6488 - val_loss: 0.8254 - val_roc_auc: 0.7018
Epoch 3/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.8341 - loss: 0.4232 - roc_auc: 0.9019 - val_accuracy: 0.6339 - val_loss: 1.0379 - val_roc_auc: 0.6874
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

=== Small LSTM Validation Performance Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Small LSTM,0.7165,0.6657,0.6667,0.6207,0.8082,0.7021



Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.53      0.62       518
           1       0.62      0.81      0.70       490

    accuracy                           0.67      1008
   macro avg       0.68      0.67      0.66      1008
weighted avg       0.69      0.67      0.66      1008



In [12]:
import pandas as pd

# 1. Combine the results of the three main models into a single DataFrame
final_comparison_df = pd.DataFrame([
    dummy_results,
    lr_results,
    lstm_results
])

# 2. Display the final sorted table rounded to 4 decimals
print("=== Final Models Comparison Table ===")
display(final_comparison_df.round(4))

=== Final Models Comparison Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Dummy Classifier,0.5238,0.4986,0.5238,0.5099,0.5245,0.5171
1,TF-IDF + Logistic Regression,0.7983,0.7846,0.7262,0.7090,0.7408,0.7246
2,Small LSTM,0.7165,0.6657,0.6667,0.6207,0.8082,0.7021


### 📊 Model Comparison Table

| Model | ROC-AUC | PR-AUC | Accuracy | Precision | Recall | F1 |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Dummy Classifier** | 0.5112 | 0.5058 | 0.5112 | 0.5113 | 0.5092 | 0.5102 |
| **TF-IDF + Logistic Regression** | 0.7861 | 0.7815 | 0.6984 | 0.7091 | 0.6728 | 0.6905 |
| **Small LSTM** | 0.7918 | 0.7960 | 0.7320 | 0.7601 | 0.6933 | 0.7251 |

---

### 🔑 Key Findings
* **Baseline Validation:** The `Dummy Classifier` establishes a random baseline (~0.50), confirming a fair and unbiased split.
* **Logistic Regression:** Achieved strong performance (`ROC-AUC: 0.7861`), proving that linear models work exceptionally well as a robust baseline for text classification.
* **LSTM Superiority:** The `Small LSTM` achieved the highest performance across all primary metrics (`ROC-AUC: 0.7918`, `PR-AUC: 0.7960`, and `F1-Score: 0.7251`), demonstrating its ability to capture deeper sequential patterns and context in the text data.

In [13]:
import pandas as pd
import numpy as np

# 1. Create an error analysis DataFrame using the Validation Set
error_df = pd.DataFrame({
    "text": val_df["text"],
    "true_label": val_df["label"],
    "predicted_label": val_preds_lstm,
    "bias_probability": val_probs_lstm
})

# Add dataset_id if it exists in your validation data
if "dataset_id" in val_df.columns:
    error_df["dataset_id"] = val_df["dataset_id"]

# 2. Classify error types
error_df["error_type"] = np.select(
    [
        (error_df["true_label"] == 0) & (error_df["predicted_label"] == 1),
        (error_df["true_label"] == 1) & (error_df["predicted_label"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

# 3. Display overall error distribution counts
print("=== Overall Error Distribution ===")
display(error_df["error_type"].value_counts().to_frame("Count"))

# 4. Error distribution by source (if dataset_id exists)
if "dataset_id" in error_df.columns:
    print("\n=== Error Distribution by Source (dataset_id) ===")
    source_errors = pd.crosstab(error_df["dataset_id"], error_df["error_type"])
    display(source_errors)

=== Overall Error Distribution ===


,Count
error_type,
Correct,672
False Positive,242
False Negative,94



=== Error Distribution by Source (dataset_id) ===


error_type,Correct,False Negative,False Positive
dataset_id,,,
75,626,91,242
110,46,3,0


In [14]:
import pickle
from pathlib import Path

# Locate the project root dynamically
project_root = Path.cwd()

if project_root.name == "modeling":
    project_root = project_root.parent.parent
elif project_root.name == "notebooks":
    project_root = project_root.parent

# Define models directory and path
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Define file paths
tokenizer_path = models_dir / "tokenizer_racial_bias.pkl"
model_path = models_dir / "lstm_racial_bias_model.keras"

# 1. Save Tokenizer configuration
with open(tokenizer_path, 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# 2. Save Trained LSTM Model
model_lstm.save(model_path)

print(f"Model saved to: {model_path}")
print(f"Tokenizer saved to: {tokenizer_path}")

Model saved to: /Users/sarahaltaleb/Documents/GitHub/BiasLens/models/lstm_racial_bias_model.keras
Tokenizer saved to: /Users/sarahaltaleb/Documents/GitHub/BiasLens/models/tokenizer_racial_bias.pkl


In [15]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_bias(text, model=model_lstm, tokenizer=tokenizer, max_len=150, threshold=0.5):
    """
    Predicts bias probability for a given raw text input.
    """
    # 1. Preprocess and tokenize input text
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(sequence, maxlen=max_len, padding='post', truncating='post')

    # 2. Generate probability score
    probability = float(model.predict(padded_sequence, verbose=0)[0][0])

    # 3. Assign label based on threshold
    is_biased = probability >= threshold

    return {
        "text": text,
        "probability": round(probability, 4),
        "label": "Biased" if is_biased else "Unbiased"
    }

# Example usage test
sample_text = "This is a sample text to test the bias prediction pipeline."
print(predict_bias(sample_text))

{'text': 'This is a sample text to test the bias prediction pipeline.', 'probability': 0.8301, 'label': 'Biased'}


In [16]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

# 1. Group-aware CV on training data (not the held-out Validation / Test).
cv_strategy = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
X_train = train_df["text"]
y_train = train_df["label"]
train_groups = train_df["split_group"]

# 2. Verify no ID group OR normalized text is shared across any CV fold.
print("=== Cross-Validation Leakage Check ===")
for fold, (train_idx, val_idx) in enumerate(
    cv_strategy.split(X_train, y_train, groups=train_groups), start=1
):
    g_train = set(train_groups.iloc[train_idx])
    g_val = set(train_groups.iloc[val_idx])
    text_train = set(train_df["normalized_text"].iloc[train_idx])
    text_val = set(train_df["normalized_text"].iloc[val_idx])
    print(f"Fold {fold}: {len(g_train & g_val)} shared groups, "
          f"{len(text_train & text_val)} shared normalized texts")
    assert not (g_train & g_val), f"ID-group leakage in Fold {fold}!"
    assert not (text_train & text_val), f"Text leakage in Fold {fold}!"

print("All group-aware CV folds verified successfully!")

=== Cross-Validation Leakage Check ===
Fold 1: 0 shared groups, 0 shared normalized texts
Fold 2: 0 shared groups, 0 shared normalized texts
Fold 3: 0 shared groups, 0 shared normalized texts
All group-aware CV folds verified successfully!


## LinearSVC

In [17]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, accuracy_score, precision_recall_fscore_support
import pandas as pd

# 1. Evaluate LinearSVC on validation data
# Predict probabilities or decision function values for AUC metrics
y_score = final_svc_pipeline.decision_function(X_val)
y_pred = final_svc_pipeline.predict(X_val)

# 2. Calculate Evaluation Metrics
roc_auc = roc_auc_score(y_val, y_score)
precision_curve, recall_curve, _ = precision_recall_curve(y_val, y_score)
pr_auc = auc(recall_curve, precision_curve)
acc = accuracy_score(y_val, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred, average='weighted')

# 3. Save LinearSVC results as a dictionary matching the existing format
svc_results = {
    'Model': 'TF-IDF + LinearSVC',
    'ROC-AUC': roc_auc,
    'PR-AUC': pr_auc,
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1': f1
}

# 4. Combine all model results into a single DataFrame
final_comparison_df = pd.DataFrame([
    dummy_results,
    lr_results,
    svc_results,    # Added LinearSVC here
    lstm_results
])

# 5. Display the final sorted table rounded to 4 decimals
print("=== Final Models Comparison Table ===")
display(final_comparison_df.round(4))

NameError: name 'final_svc_pipeline' is not defined

### Full Train LinearSVC

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, accuracy_score, precision_recall_fscore_support

# 1. Combine Training and Validation sets for full training
# (Or use the entire X_train dataset if it represents your full dataset)
X_full_train = pd.concat([X_train, X_val]) if 'X_val' in locals() else X_train
y_full_train = pd.concat([y_train, y_val]) if 'y_val' in locals() else y_train

# 2. Extract best parameters from GridSearchCV (or set optimal parameters manually)
best_params = grid_search.best_params_ if 'grid_search' in locals() else {
    'tfidf__ngram_range': (1, 2),
    'tfidf__min_df': 2,
    'svc__C': 1.0,
    'svc__class_weight': 'balanced'
}

# 3. Define and train the final LinearSVC pipeline on the full dataset
final_svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        sublinear_tf=True,
        ngram_range=best_params.get('tfidf__ngram_range', (1, 2)),
        min_df=best_params.get('tfidf__min_df', 2)
    )),
    ('svc', LinearSVC(
        C=best_params.get('svc__C', 1.0),
        class_weight=best_params.get('svc__class_weight', 'balanced'),
        random_state=42,
        max_iter=2000
    ))
])

# Fit on full training set
final_svc_pipeline.fit(X_full_train, y_full_train)

# 4. Evaluate on Full Training Set
y_score_full = final_svc_pipeline.decision_function(X_full_train)
y_pred_full = final_svc_pipeline.predict(X_full_train)

# Calculate evaluation metrics
roc_auc_full = roc_auc_score(y_full_train, y_score_full)
precision_curve, recall_curve, _ = precision_recall_curve(y_full_train, y_score_full)
pr_auc_full = auc(recall_curve, precision_curve)
acc_full = accuracy_score(y_full_train, y_pred_full)
prec_full, rec_full, f1_full, _ = precision_recall_fscore_support(y_full_train, y_pred_full, average='weighted')

# 5. Format results as dictionary for comparison table
svc_full_results = {
    'Model': 'TF-IDF + LinearSVC (Full Train)',
    'ROC-AUC': roc_auc_full,
    'PR-AUC': pr_auc_full,
    'Accuracy': acc_full,
    'Precision': prec_full,
    'Recall': rec_full,
    'F1': f1_full
}

# 6. Display in comparison table format
final_comparison_df = pd.DataFrame([
    svc_full_results
])

print("=== Full Training Set Results ===")
display(final_comparison_df.round(4))

=== Full Training Set Results ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,TF-IDF + LinearSVC (Full Train),0.9934,0.9932,0.965,0.965,0.965,0.965


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    auc,
    accuracy_score,
    precision_recall_fscore_support
)

# 1. Combine Training and Validation sets for full dataset training
X_full_train = pd.concat([X_train, X_val]) if 'X_val' in locals() else X_train
y_full_train = pd.concat([y_train, y_val]) if 'y_val' in locals() else y_train

# 2. Extract best parameters or set optimal defaults
best_params = grid_search.best_params_ if 'grid_search' in locals() else {
    'tfidf__ngram_range': (1, 2),
    'tfidf__min_df': 2,
    'svc__C': 1.0,
    'svc__class_weight': 'balanced'
}

# 3. Build and train final LinearSVC pipeline on full dataset
final_svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        sublinear_tf=True,
        ngram_range=best_params.get('tfidf__ngram_range', (1, 2)),
        min_df=best_params.get('tfidf__min_df', 2)
    )),
    ('svc', LinearSVC(
        C=best_params.get('svc__C', 1.0),
        class_weight=best_params.get('svc__class_weight', 'balanced'),
        random_state=42,
        max_iter=2000
    ))
])

final_svc_pipeline.fit(X_full_train, y_full_train)

# 4. Evaluate metrics on Full Training Set
y_score_full = final_svc_pipeline.decision_function(X_full_train)
y_pred_full = final_svc_pipeline.predict(X_full_train)

roc_auc_full = roc_auc_score(y_full_train, y_score_full)
precision_curve, recall_curve, _ = precision_recall_curve(y_full_train, y_score_full)
pr_auc_full = auc(recall_curve, precision_curve)
acc_full = accuracy_score(y_full_train, y_pred_full)
prec_full, rec_full, f1_full, _ = precision_recall_fscore_support(y_full_train, y_pred_full, average='weighted')

# 5. Format dictionary for Full Train results
svc_full_results = {
    'Model': 'TF-IDF + LinearSVC (Full Train)',
    'ROC-AUC': roc_auc_full,
    'PR-AUC': pr_auc_full,
    'Accuracy': acc_full,
    'Precision': prec_full,
    'Recall': rec_full,
    'F1': f1_full
}

# 6. Combine all models into the Final Comparison Table
final_comparison_df = pd.DataFrame([
    dummy_results,
    lr_results,
    svc_full_results,
    lstm_results,
    lstm_full_results
])

# 7. Display the complete sorted table rounded to 4 decimals
print("=== Final Models Comparison Table ===")
display(final_comparison_df.set_index("Model").round(4))

=== Final Models Comparison Table ===


,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
Model,,,,,,
Dummy Classifier,0.5112,0.5058,0.5112,0.5113,0.5092,0.5102
TF-IDF + Logistic Regression,0.7841,0.7801,0.7014,0.7091,0.6830,0.6958
TF-IDF + LinearSVC (Full Train),0.9934,0.9932,0.9650,0.9650,0.9650,0.9650
Small LSTM,0.7992,0.7971,0.7270,0.7478,0.6851,0.7150
Small LSTM (Full Train),0.9939,0.9941,0.9691,0.9687,0.9696,0.9691


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 1. Predict on Training Set
y_train_pred = final_svc_pipeline.predict(X_train)
y_train_score = final_svc_pipeline.decision_function(X_train)

# 2. Predict on Validation Set
y_val_pred = final_svc_pipeline.predict(X_val)
y_val_score = final_svc_pipeline.decision_function(X_val)

# 3. Calculate metrics for Training Set
train_acc = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted')
train_roc = roc_auc_score(y_train, y_train_score)

# 4. Calculate metrics for Validation Set
val_acc = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average='weighted')
val_roc = roc_auc_score(y_val, y_val_score)

# 5. Create a comparison DataFrame to check for Overfitting
overfitting_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-Score (Weighted)', 'ROC-AUC'],
    'Training Set': [train_acc, train_f1, train_roc],
    'Validation Set': [val_acc, val_f1, val_roc]
})

# Add a difference column to easily spot gaps
overfitting_df['Gap (Train - Val)'] = overfitting_df['Training Set'] - overfitting_df['Validation Set']

# 6. Display the table
print("=== Overfitting Analysis Table ===")
display(overfitting_df.set_index('Metric').round(4))

=== Overfitting Analysis Table ===


,Training Set,Validation Set,Gap (Train - Val)
Metric,,,
Accuracy,0.9645,0.9693,-0.0048
F1-Score (Weighted),0.9645,0.9693,-0.0048
ROC-AUC,0.9931,0.9963,-0.0032


In [ ]:
# 1. Check for overlapping samples (exact text duplicates) between Train and Validation sets
train_texts = set(X_train)
val_texts = set(X_val)

overlap = train_texts.intersection(val_texts)

print(f"Number of training samples: {len(X_train)}")
print(f"Number of validation samples: {len(X_val)}")
print(f"Number of overlapping samples between Train and Val: {len(overlap)}")

if len(overlap) == 0:
    assert not (set(train_df["split_group"]) & set(val_df["split_group"]))
    print("No normalized-text or source-specific ID-group overlap in Train / Validation.")
else:
    print(f"⚠️ Warning: Found {len(overlap)} overlapping samples between train and validation.")

### Full Train LSTM

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Combine training and validation sequences and labels for full training
X_full_pad = np.concatenate([X_train_pad, X_val_pad])
y_full = pd.concat([train_df["label"], val_df["label"]]).values

# 2. Re-build the LSTM model architecture for full training
model_lstm_full = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=64, mask_zero=True),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model_lstm_full.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')]
)

# 3. Train the model on the full dataset
model_lstm_full.fit(
    X_full_pad,
    y_full,
    batch_size=64,
    epochs=5,          # Adjust epochs if needed based on previous training behavior
    verbose=1
)

# 4. Evaluate the full-trained LSTM on the full dataset & store metrics in a dictionary
val_probs_lstm_full = model_lstm_full.predict(X_full_pad).flatten()
val_preds_lstm_full = (val_probs_lstm_full >= 0.5).astype(int)

lstm_full_results = {
    "Model": "Small LSTM (Full Train)",
    "ROC-AUC": roc_auc_score(y_full, val_probs_lstm_full),
    "PR-AUC": average_precision_score(y_full, val_probs_lstm_full),
    "Accuracy": accuracy_score(y_full, val_preds_lstm_full),
    "Precision": precision_score(y_full, val_preds_lstm_full, zero_division=0),
    "Recall": recall_score(y_full, val_preds_lstm_full, zero_division=0),
    "F1": f1_score(y_full, val_preds_lstm_full, zero_division=0)
}

# 5. Combine all models into a final structured comparison DataFrame
final_comparison_df = pd.DataFrame([
    dummy_results,
    lr_results,
    svc_full_results,
    lstm_results,
    lstm_full_results
])

# 6. Display the final sorted and formatted comparison table
print("\n=== Final Models Comparison Table (Including Full Train LSTM) ===")
display(final_comparison_df.set_index("Model").round(4))

Epoch 1/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 37s 214ms/step - accuracy: 0.6543 - loss: 0.6112 - roc_auc: 0.7311
Epoch 2/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 24s 176ms/step - accuracy: 0.8012 - loss: 0.4451 - roc_auc: 0.8758
Epoch 3/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 29s 209ms/step - accuracy: 0.8654 - loss: 0.3301 - roc_auc: 0.9334
Epoch 4/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 30s 214ms/step - accuracy: 0.9114 - loss: 0.2381 - roc_auc: 0.9654
Epoch 5/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 26s 185ms/step - accuracy: 0.9355 - loss: 0.1741 - roc_auc: 0.9813
276/276 ━━━━━━━━━━━━━━━━━━━━ 11s 39ms/step

=== Final Models Comparison Table (Including Full Train LSTM) ===


,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
Model,,,,,,
Dummy Classifier,0.5112,0.5058,0.5112,0.5113,0.5092,0.5102
TF-IDF + Logistic Regression,0.7841,0.7801,0.7014,0.7091,0.6830,0.6958
TF-IDF + LinearSVC (Full Train),0.9934,0.9932,0.9650,0.9650,0.9650,0.9650
Small LSTM,0.7992,0.7971,0.7270,0.7478,0.6851,0.7150
Small LSTM (Full Train),0.9938,0.9942,0.9670,0.9628,0.9714,0.9671


In [ ]:
import pandas as pd
import numpy as np

# --- 1. Build Error Analysis DataFrame on Validation Set ---
error_df = pd.DataFrame({
    "text": val_df["text"],
    "true_label": val_df["label"],
    "predicted_label": val_preds_lstm,
    "probability": val_probs_lstm
})

# --- 2. Classify Predictions into Correct, False Positive, or False Negative ---
error_df["error_type"] = np.select(
    [
        (error_df["true_label"] == 0) & (error_df["predicted_label"] == 1),
        (error_df["true_label"] == 1) & (error_df["predicted_label"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

# --- 3. Display Overall Error Counts ---
print("=== Overall Error Distribution ===")
display(error_df["error_type"].value_counts().to_frame("Count"))

# --- 4. Extract Incorrect Predictions for Analysis ---
incorrect_df = error_df[error_df["error_type"] != "Correct"].copy()

# Add text length to check if short/long texts are harder
incorrect_df["text_length"] = incorrect_df["text"].apply(len)

print(f"\nTotal incorrect predictions to analyze: {len(incorrect_df)}")

# --- 5. Display a sample of false positives and false negatives for presentation ---
print("\n--- Sample False Positives (Predicted Biased, Actually Safe) ---")
fp_sample = incorrect_df[incorrect_df["error_type"] == "False Positive"][["text", "true_label", "predicted_label", "probability"]].head(2)
display(fp_sample)

print("\n--- Sample False Negatives (Predicted Safe, Actually Biased) ---")
fn_sample = incorrect_df[incorrect_df["error_type"] == "False Negative"][["text", "true_label", "predicted_label", "probability"]].head(2)
display(fn_sample)

=== Overall Error Distribution ===


,Count
error_type,
Correct,711
False Negative,154
False Positive,113



Total incorrect predictions to analyze: 267

--- Sample False Positives (Predicted Biased, Actually Safe) ---


,text,true_label,predicted_label,probability
3736,yeah i was curious why they included it too br...,0,1,0.649013
2748,if im remembering this correctly satan in juda...,0,1,0.564652



--- Sample False Negatives (Predicted Safe, Actually Biased) ---


,text,true_label,predicted_label,probability
5806,nta i think your aunt likely just forgot but i...,1,0,0.219794
9141,some homosexuals are undoubtedly cowards he wo...,1,0,0.081229


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# --- 1. LinearSVC Overfitting Metrics ---
train_preds_svc = final_svc_pipeline.predict(X_train)
val_preds_svc = final_svc_pipeline.predict(X_val)

train_probs_svc = final_svc_pipeline.decision_function(X_train)
val_probs_svc = final_svc_pipeline.decision_function(X_val)

svc_train_acc = accuracy_score(y_train, train_preds_svc)
svc_val_acc = accuracy_score(y_val, val_preds_svc)
svc_train_f1 = f1_score(y_train, train_preds_svc, average='weighted')
svc_val_f1 = f1_score(y_val, val_preds_svc, average='weighted')
svc_train_roc = roc_auc_score(y_train, train_probs_svc)
svc_val_roc = roc_auc_score(y_val, val_probs_svc)

# --- 2. Small LSTM Overfitting Metrics ---
train_probs_lstm = model_lstm.predict(X_train_pad).flatten()
val_probs_lstm = model_lstm.predict(X_val_pad).flatten()

train_preds_lstm = (train_probs_lstm >= 0.5).astype(int)
val_preds_lstm = (val_probs_lstm >= 0.5).astype(int)

lstm_train_acc = accuracy_score(train_df["label"], train_preds_lstm)
lstm_val_acc = accuracy_score(val_df["label"], val_preds_lstm)
lstm_train_f1 = f1_score(train_df["label"], train_preds_lstm, average='weighted')
lstm_val_f1 = f1_score(val_df["label"], val_preds_lstm, average='weighted')
lstm_train_roc = roc_auc_score(train_df["label"], train_probs_lstm)
lstm_val_roc = roc_auc_score(val_df["label"], val_probs_lstm)

# --- 3. Logistic Regression Overfitting Metrics ---
train_preds_lr = lr_pipeline.predict(X_train)
val_preds_lr = lr_pipeline.predict(X_val)

train_probs_lr = lr_pipeline.predict_proba(X_train)[:, 1]
val_probs_lr = lr_pipeline.predict_proba(X_val)[:, 1]

lr_train_acc = accuracy_score(y_train, train_preds_lr)
lr_val_acc = accuracy_score(y_val, val_preds_lr)
lr_train_f1 = f1_score(y_train, train_preds_lr, average='weighted')
lr_val_f1 = f1_score(y_val, val_preds_lr, average='weighted')
lr_train_roc = roc_auc_score(y_train, train_probs_lr)
lr_val_roc = roc_auc_score(y_val, val_probs_lr)

# --- 4. Combine into a Unified Summary DataFrame ---
overfitting_summary_df = pd.DataFrame({
    'Model': ['LinearSVC', 'Small LSTM', 'Logistic Regression'],
    'Train Accuracy': [svc_train_acc, lstm_train_acc, lr_train_acc],
    'Val Accuracy': [svc_val_acc, lstm_val_acc, lr_val_acc],
    'Acc Gap': [svc_train_acc - svc_val_acc, lstm_train_acc - lstm_val_acc, lr_train_acc - lr_val_acc],

    'Train F1': [svc_train_f1, lstm_train_f1, lr_train_f1],
    'Val F1': [svc_val_f1, lstm_val_f1, lr_val_f1],
    'F1 Gap': [svc_train_f1 - svc_val_f1, lstm_train_f1 - lstm_val_f1, lr_train_f1 - lr_val_f1],

    'Train ROC-AUC': [svc_train_roc, lstm_train_roc, lr_train_roc],
    'Val ROC-AUC': [svc_val_roc, lstm_val_roc, lr_val_roc],
    'ROC Gap': [svc_train_roc - svc_val_roc, lstm_train_roc - lstm_val_roc, lr_train_roc - lr_val_roc]
})

# --- 5. Display Final Unified Overfitting Table ---
print("=== Unified Models Overfitting Comparison Table ===")
display(overfitting_summary_df.set_index("Model").round(4))

245/245 ━━━━━━━━━━━━━━━━━━━━ 21s 83ms/step
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step
=== Unified Models Overfitting Comparison Table ===


,Train Accuracy,Val Accuracy,Acc Gap,Train F1,Val F1,F1 Gap,Train ROC-AUC,Val ROC-AUC,ROC Gap
Model,,,,,,,,,
LinearSVC,0.9645,0.9693,-0.0048,0.9645,0.9693,-0.0048,0.9931,0.9963,-0.0032
Small LSTM,0.8375,0.7270,0.1105,0.8373,0.7265,0.1108,0.9088,0.7992,0.1096
Logistic Regression,0.7934,0.7014,0.0920,0.7934,0.7013,0.0921,0.8670,0.7841,0.0829


### 🔑 Key Findings

* **Baseline Validation:** The `Dummy Classifier` establishes a random baseline (~0.52), confirming a fair and unbiased split.
* **Logistic Regression Superiority:** Achieved the highest overall performance across primary metrics (`ROC-AUC: 0.7983`, `PR-AUC: 0.7846`, and `F1-Score: 0.7246`), proving that linear models work exceptionally well as a robust baseline for text classification.
* **LSTM Performance:** The `Small LSTM` achieved high sensitivity/recall (`Recall: 0.8082`), but lower precision (`Precision: 0.6207`) and overall ROC-AUC (`0.7165`) compared to the TF-IDF model.